# Simple Linear Regressions

In [ ]:
# Init

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load data

df=pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%202/data/FuelConsumptionCo2.csv")

In [ ]:
df.describe()

In [ ]:
# Feature selecting to mmeasure CO2 emissions

select_df = df[['ENGINESIZE','CYLINDERS','FUELCONSUMPTION_COMB','CO2EMISSIONS']]

In [ ]:
# Visualisation using histogram

select_df.hist()
plt.show()

In [ ]:
# Visualisation using scatter

plt.scatter(select_df.FUELCONSUMPTION_COMB, select_df.CO2EMISSIONS)
plt.xlabel("Combined Fuel Consumption (Highway and City)")
plt.ylabel("CO2 Emissions")
plt.show()


In [ ]:
# We will use Engine Size to predict CO2 Emissions using a simple linear regression
# First we must extrat the input features

x = select_df.ENGINESIZE.to_numpy()
Y = select_df.CO2EMISSIONS.to_numpy()

In [ ]:
# Split data into training and test datasets

from sklearn.model_selection import train_test_split

x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size=0.2, random_state=42)

In [ ]:
# Building simple linear regression

from sklearn import linear_model

reg_obj = linear_model.LinearRegression()

# Train the model 

reg_obj.fit(x_train.reshape(-1, 1), Y_train) # We reshape it because sklearn models expect a 2D array instead of a 1D for input.

# Print Coefficients

print ('Coefficients: ', reg_obj.coef_[0])
print ('Intercept: ', reg_obj.intercept_)

In [ ]:
# Visualise model output

plt.scatter(x_train, Y_train)
plt.plot(x_train, reg_obj.coef_[0] * x_train + reg_obj.intercept_, 'r')
plt.xlabel("Engine size")
plt.ylabel("Emission")

In [ ]:
# Model evaluation
# Can use Mean Absolute Error, MSE, Root MSE and R2

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_hat = reg_obj.predict(x_test.reshape(-1, 1))

# Evaluation
print("Mean absolute error: %.2f" % mean_absolute_error(Y_test, y_hat))
print("Mean squared error: %.2f" % mean_squared_error(Y_test, y_hat))
print("Root mean squared error: %.2f" % np.sqrt(mean_squared_error(Y_test, y_hat)))
print("R2-score: %.2f" % r2_score(Y_test, y_hat))

# Multiple Linear Regression

In [ ]:
# In the interest of time we are just going to use the numerical values but we could have transformed them

df = df.drop(['MODELYEAR', 'MAKE', 'MODEL', 'VEHICLECLASS', 'TRANSMISSION', 'FUELTYPE',],axis=1)

In [ ]:
# Check for multicollineariarity 
# Nothing is perfect which is the assumption of OLS by there is 0.93 between cylinders and engine size which makes sense.
df.corr()

# Apparantly we are removing correlations above 85

df = df.drop(['CYLINDERS', 'FUELCONSUMPTION_CITY', 'FUELCONSUMPTION_HWY','FUELCONSUMPTION_COMB',],axis=1)

In [ ]:
# Visually checking correlations everything seems good but some relationships seem non-linear.

axes = pd.plotting.scatter_matrix(df, alpha=0.2)
# need to rotate axis labels so we can read them
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(90)
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')

plt.tight_layout()
plt.gcf().subplots_adjust(wspace=0, hspace=0)
plt.show()

In [ ]:
# Extracting input features

X = df.iloc[:,[0,1]].to_numpy() # Columns 0 and 1
y = df.iloc[:,[2]].to_numpy() # Column 2

In [ ]:
# Pre-process the selected values to not have model favour one variable to another based on magnitude.
# This is overly simplified and has pitfuls if done in practise.
from sklearn import preprocessing

std_scaler = preprocessing.StandardScaler() 
X_std = std_scaler.fit_transform(X) # I think this subtracts the mean and divdes it by the standard deviation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_std,y,test_size=0.2,random_state=42)

In [ ]:
# create a model object
regressor = linear_model.LinearRegression()

# train the model in the training data
regressor.fit(X_train, y_train)

# Print the coefficients
coef_ =  regressor.coef_
intercept_ = regressor.intercept_

print ('Coefficients: ',coef_)
print ('Intercept: ',intercept_)

In [ ]:
# Unstandardised values

# Get the standard scaler's mean and standard deviation parameters
means_ = std_scaler.mean_
std_devs_ = np.sqrt(std_scaler.var_)

# The least squares parameters can be calculated relative to the original, unstandardized feature space as:
coef_original = coef_ / std_devs_
intercept_original = intercept_ - np.sum((means_ * coef_) / std_devs_)

print ('Coefficients: ', coef_original)
print ('Intercept: ', intercept_original)


# Logistic Regression (Machine learning)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Assume a telecommunication wants to know the probability of a person leaving.
churn_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/ChurnData.csv")

In [ ]:
# Data preprocessing
# Using a subset

churn_df = churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip', 'churn']]
churn_df['churn'] = churn_df['churn'].astype('int')

In [ ]:
# Extract input features
X= np.asarray(churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip']])
y = np.asarray(churn_df['churn'])

In [ ]:
# Normalise the input

X_norm = StandardScaler().fit(X).transform(X)

In [ ]:
# Splitting dataset

X_train, X_test, y_train, y_test = train_test_split( X_norm, y, test_size=0.2, random_state=4)

In [ ]:
# Model

LR = LogisticRegression().fit(X_train, y_train)
yhat = LR.predict(X_test)

In [ ]:
# Probablity
# First column is probablity of class o and second column is probablity of class 1.
yhat_prob = LR.predict_proba(X_test)

In [ ]:
# Check the impact of variables on predicting a class of 1

coefficients = pd.Series(LR.coef_[0], index=churn_df.columns[:-1])
coefficients.sort_values().plot(kind='barh')
plt.title("Feature Coefficients in Logistic Regression Churn Model")
plt.xlabel("Coefficient Value")
plt.show()

In [32]:
# Model Evaluation

log_loss(y_test, yhat_prob)

0.6257718410257235